# Water Detection for Sentinel-1 and Sentinel-2 Imagery

This notebook performs water detection on both Sentinel-1 (SAR) and Sentinel-2 (optical) imagery. It automatically detects the sensor type and applies the appropriate water detection method.

## What is Water Detection?

Water detection identifies water bodies (lakes, rivers, oceans, flooded areas) in satellite imagery using sensor-specific methods:

### Sentinel-1 (SAR - Synthetic Aperture Radar)
- Uses backscatter intensity: Water has low backscatter (dark in SAR imagery)
- Method: Threshold on VV polarization backscatter (typically < -15 dB)
- Advantages: Works through clouds, day/night imaging

### Sentinel-2 (Optical)
- Uses spectral indices: NDWI (Normalized Difference Water Index)
- Method: Threshold on NDWI values (typically > 0.0 or > 0.3)
- Advantages: Higher spatial resolution, color information

## Parameters

This notebook has been automatically configured with the following parameters:

- **Collection**: {{STAC_COLLECTION_NAME}}
- **STAC Item**: {{STAC_ITEM_LINK}}
- **AOI (Area of Interest)**: {{AOI}} *(optional - if not provided, a 1000x1000 pixel sample will be used)*

## Workflow

1. Load the STAC item
2. Detect sensor type (Sentinel-1 or Sentinel-2)
3. Access appropriate bands (VV for S1, Green/NIR for S2)
4. Apply sensor-specific water detection method
5. Visualize the results

## Step 1: Import Required Libraries

In [ ]:
import pystac
import rasterio
from rasterio.windows import Window
from rasterio.mask import mask
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import json
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

print("Libraries imported successfully!")

## Step 2: Determine Area of Interest (AOI)

Determine the area to process. If an AOI is provided, clip to that geometry. Otherwise, extract a 1000x1000 pixel sample from the center of the image.

In [ ]:
import json

AOI = {
    "type": "FeatureCollection",
    "features": [
        {
            "id": "4c58839e-6e98-4770-bd38-34c5de05c5e0",
            "type": "Feature",
            "geometry": {
                "type": "Polygon",
                "coordinates": [
                    [
                        [-3.315606699, 54.158180301],
                        [-3.315606699, 54.063450114],
                        [-3.165173184, 54.063450114],
                        [-3.165173184, 54.158180301],
                        [-3.315606699, 54.158180301],
                    ]
                ],
            },
            "properties": {"mode": "rectangle"},
        }
    ],
}
AOI = json.dumps(AOI)
manual_stac_item_url = "https://eodatahub.org.uk/api/catalogue/stac/catalogs/public/catalogs/ceda-stac-catalogue/collections/sentinel1/items/neodc.sentinel1a.data.IW.L1_SLC.IPF_v3.2024.10.21.S1A_IW_SLC__1SDV_20241021T063900_20241021T063928_056197_06E111_E252"
manual_stac_collection_name = "sentinel1"


In [ ]:
# AOI parameter - must be GeoJSON geometry (Polygon, MultiPolygon, etc.)
# Using triple quotes to safely handle JSON strings with quotes
aoi_param = """{{AOI}}""".strip()


# Default window size if no AOI provided
DEFAULT_WINDOW_SIZE = 1000  # pixels

aoi_geometry = None
clip_window = None
use_windowed_read = False

# Check if AOI is provided
if aoi_param and aoi_param.lower() not in ["", "none", "null"]:
    try:
        # Parse as JSON
        aoi_data = json.loads(aoi_param)

        # Extract geometry from GeoJSON structure
        if aoi_data.get("type") == "FeatureCollection":
            # Extract first geometry from FeatureCollection
            if aoi_data.get("features") and len(aoi_data["features"]) > 0:
                aoi_geometry = aoi_data["features"][0].get("geometry")
        elif aoi_data.get("type") == "Feature":
            # Extract geometry from Feature
            aoi_geometry = aoi_data.get("geometry")
        elif aoi_data.get("type") in ["Polygon", "MultiPolygon", "Point", "LineString"]:
            # Direct geometry object
            aoi_geometry = aoi_data
        else:
            raise ValueError(f"Unsupported GeoJSON type: {aoi_data.get('type')}")

        # Validate geometry was extracted
        if aoi_geometry and aoi_geometry.get("type"):
            print(f"AOI provided: {aoi_geometry['type']} geometry")
            print("Will clip raster to AOI geometry")
        else:
            raise ValueError("Could not extract geometry from GeoJSON")

    except (json.JSONDecodeError, ValueError, KeyError) as e:
        print(f"Warning: Could not parse AOI as GeoJSON: {e}")
        print("Falling back to default 1000x1000 pixel window")
        aoi_geometry = None
else:
    print("No AOI provided, using default 1000x1000 pixel window")

# Set up windowed read if no AOI geometry
if aoi_geometry is None:
    use_windowed_read = True
    print(
        f"Will extract {DEFAULT_WINDOW_SIZE}x{DEFAULT_WINDOW_SIZE} pixel window from center"
    )

In [ ]:
# STAC item URL - automatically populated from selected dataset
stac_item_url = "{{STAC_ITEM_LINK}}"
stac_collection_name = "{{STAC_COLLECTION_NAME}}"

stac_item_url = manual_stac_item_url
stac_collection_name = manual_stac_collection_name

try:
    # Load the STAC item
    item = pystac.Item.from_file(stac_item_url)

    print(f"Successfully loaded STAC item: {item.id}")
    print(f"Collection: {stac_collection_name}")
    print(f"Date: {item.datetime}")
    print(f"Geometry: {item.geometry}")

except Exception as e:
    print(f"Error loading STAC item: {e}")
    raise

## Step 3: Detect Sensor Type

Automatically detect whether this is Sentinel-1 (SAR) or Sentinel-2 (optical) imagery by checking the collection name, STAC properties, and available bands.

In [ ]:
def detect_sensor_type(stac_collection_name):
    collection_lower = stac_collection_name.lower()
    s1_keywords = ["sentinel-1", "s1", "sar", "sentinel1"]
    s2_keywords = ["sentinel-2", "s2", "sentinel2"]
    if any(keyword in collection_lower for keyword in s1_keywords):
        print("  Method: SAR backscatter thresholding")
        return "S1"
    elif any(keyword in collection_lower for keyword in s2_keywords):
        print("  Method: NDWI (Normalized Difference Water Index)")
        return "S2"
    else:
        print(
            "Warning: Could not automatically detect sensor type. Defaulting to Sentinel-2."
        )
        return None


sensor_type = detect_sensor_type(stac_collection_name=stac_collection_name)

## Step 4: Access Bands

Access the appropriate bands based on sensor type:
- **Sentinel-1**: VV polarization (primary), optionally VH
- **Sentinel-2**: Green (B03) and NIR (B08) bands

In [ ]:
try:
    if sensor_type == "S1":
        # Sentinel-1: Look for VV polarization
        vv_asset = None
        vh_asset = None
        cog_asset = None
        vv_band_index = None
        vh_band_index = None

        # Look for separate assets
        for asset_key, asset in item.assets.items():
            key_upper = asset_key.upper()
            if "VV" in key_upper and vv_asset is None:
                vv_asset = asset
                print(f"Found VV polarization: {asset_key}")
            elif "VH" in key_upper and vh_asset is None:
                vh_asset = asset
                print(f"Found VH polarization: {asset_key}")

        # If not found, look for multi-band COG
        if vv_asset is None:
            for asset_key in ["cog", "data", "image", "vv", "vh"]:
                if asset_key in item.assets:
                    cog_asset = item.assets[asset_key]
                    print(f"Using multi-band asset: {asset_key}")
                    # Assume VV is first band, VH is second if available
                    vv_band_index = 1
                    vh_band_index = 2
                    break

        if vv_asset is None and cog_asset is None:
            raise ValueError("Could not find VV polarization band for Sentinel-1")

        print("\nSentinel-1 bands found successfully!")
        if vv_asset:
            print(f"  VV: {vv_asset.href}")
        if vh_asset:
            print(f"  VH: {vh_asset.href}")
        if cog_asset:
            print(f"  Multi-band COG: {cog_asset.href}")

    else:  # Sentinel-2
        # Sentinel-2: Look for Green (B03) and NIR (B08)
        green_band_asset = None
        nir_band_asset = None
        cog_asset = None
        green_band_index = 3  # B03
        nir_band_index = 8  # B08

        # Look for separate assets
        for asset_key, asset in item.assets.items():
            key_upper = asset_key.upper()
            key_lower = asset_key.lower()
            if "B03" in key_upper or (
                "green" in key_lower and "visual" not in key_lower
            ):
                green_band_asset = asset
                print(f"Found green band (B03): {asset_key}")
            if "B08" in key_upper or ("nir" in key_lower and "B08" in key_upper):
                nir_band_asset = asset
                print(f"Found NIR band (B08): {asset_key}")

        # If not found, look for multi-band COG
        if green_band_asset is None or nir_band_asset is None:
            for asset_key in ["cog", "data", "image", "reflectance", "bands"]:
                if asset_key in item.assets:
                    cog_asset = item.assets[asset_key]
                    # Try to get band indices from STAC eo:bands
                    extra = getattr(cog_asset, "extra", {}) or {}
                    eo_bands = extra.get(
                        "eo:bands", item.properties.get("eo:bands", [])
                    )
                    if eo_bands:
                        for i, b in enumerate(eo_bands):
                            name = (b.get("name") or b.get("common_name") or "").upper()
                            if "B03" in name or b.get("common_name") == "green":
                                green_band_index = i + 1
                            if "B08" in name or b.get("common_name") == "nir":
                                nir_band_index = i + 1
                    print(
                        f"Using multi-band asset '{asset_key}' (B03=band {green_band_index}, B08=band {nir_band_index})"
                    )
                    break

        if green_band_asset is None and nir_band_asset is None and cog_asset is None:
            raise ValueError(
                "Could not find green (B03) or NIR (B08) bands for Sentinel-2"
            )

        print(f"\nSentinel-2 bands found successfully!")
        if green_band_asset and nir_band_asset:
            print(f"  Green: {green_band_asset.href}")
            print(f"  NIR: {nir_band_asset.href}")
        else:
            print(f"  Multi-band COG: {cog_asset.href}")

except Exception as e:
    print(f"Error accessing bands: {e}")
    print("\nAvailable assets:")
    for asset_key in item.assets.keys():
        print(f"  - {asset_key}")
    raise

## Step 5: Read Band Data

Read the band data into numpy arrays. If an AOI was provided, the data will be clipped to that geometry. Otherwise, a 1000x1000 pixel window will be extracted from the center.

In [ ]:
def read_band_data(src, band_idx, aoi_geometry, clip_window, use_windowed_read):
    """Helper function to read band data with AOI or window support."""
    if aoi_geometry is not None:
        data, transform = mask(src, [aoi_geometry], crop=True, indexes=[band_idx])
        return data[0], transform
    elif use_windowed_read and clip_window is not None:
        return src.read(band_idx, window=clip_window), None
    elif use_windowed_read:
        return src.read(band_idx), None
    else:
        return src.read(band_idx), None


try:
    # Ensure AOI variables are initialized (in case AOI cell was not executed)
    try:
        _ = aoi_geometry
    except NameError:
        aoi_geometry = None
        clip_window = None
        use_windowed_read = False

    if sensor_type == "S1":
        # Read Sentinel-1 VV band
        if cog_asset is not None:
            with rasterio.open(cog_asset.href) as src:
                if aoi_geometry is None and use_windowed_read:
                    height, width = src.height, src.width
                    if height >= DEFAULT_WINDOW_SIZE and width >= DEFAULT_WINDOW_SIZE:
                        row_off = (height - DEFAULT_WINDOW_SIZE) // 2
                        col_off = (width - DEFAULT_WINDOW_SIZE) // 2
                        clip_window = Window(
                            col_off, row_off, DEFAULT_WINDOW_SIZE, DEFAULT_WINDOW_SIZE
                        )
                        print(
                            f"Extracting {DEFAULT_WINDOW_SIZE}x{DEFAULT_WINDOW_SIZE} pixel window"
                        )
                    else:
                        clip_window = None

                vv_data, transform = read_band_data(
                    src, vv_band_index, aoi_geometry, clip_window, use_windowed_read
                )
                profile = src.profile.copy()
                crs = src.crs
        else:
            with rasterio.open(vv_asset.href) as src:
                if aoi_geometry is None and use_windowed_read:
                    height, width = src.height, src.width
                    if height >= DEFAULT_WINDOW_SIZE and width >= DEFAULT_WINDOW_SIZE:
                        row_off = (height - DEFAULT_WINDOW_SIZE) // 2
                        col_off = (width - DEFAULT_WINDOW_SIZE) // 2
                        clip_window = Window(
                            col_off, row_off, DEFAULT_WINDOW_SIZE, DEFAULT_WINDOW_SIZE
                        )
                    else:
                        clip_window = None
                    vv_data = (
                        src.read(1, window=clip_window) if clip_window else src.read(1)
                    )
                elif aoi_geometry is not None:
                    vv_data, transform = mask(src, [aoi_geometry], crop=True)
                    vv_data = vv_data[0]
                else:
                    vv_data = src.read(1)
                profile = src.profile.copy()
                crs = src.crs

        vv_data = vv_data.astype(np.float32)
        print(f"\nSentinel-1 VV band loaded: {vv_data.shape}, dtype: {vv_data.dtype}")
        print(f"CRS: {crs}")

    else:  # Sentinel-2
        # Read Sentinel-2 Green and NIR bands
        if cog_asset is not None:
            with rasterio.open(cog_asset.href) as src:
                if aoi_geometry is None and use_windowed_read:
                    height, width = src.height, src.width
                    if height >= DEFAULT_WINDOW_SIZE and width >= DEFAULT_WINDOW_SIZE:
                        row_off = (height - DEFAULT_WINDOW_SIZE) // 2
                        col_off = (width - DEFAULT_WINDOW_SIZE) // 2
                        clip_window = Window(
                            col_off, row_off, DEFAULT_WINDOW_SIZE, DEFAULT_WINDOW_SIZE
                        )
                        print(
                            f"Extracting {DEFAULT_WINDOW_SIZE}x{DEFAULT_WINDOW_SIZE} pixel window"
                        )
                    else:
                        clip_window = None

                green_data, transform = read_band_data(
                    src, green_band_index, aoi_geometry, clip_window, use_windowed_read
                )
                nir_data, _ = read_band_data(
                    src, nir_band_index, aoi_geometry, clip_window, use_windowed_read
                )
                profile = src.profile.copy()
                crs = src.crs
        else:
            with rasterio.open(green_band_asset.href) as green_src:
                if aoi_geometry is None and use_windowed_read:
                    height, width = green_src.height, green_src.width
                    if height >= DEFAULT_WINDOW_SIZE and width >= DEFAULT_WINDOW_SIZE:
                        row_off = (height - DEFAULT_WINDOW_SIZE) // 2
                        col_off = (width - DEFAULT_WINDOW_SIZE) // 2
                        clip_window = Window(
                            col_off, row_off, DEFAULT_WINDOW_SIZE, DEFAULT_WINDOW_SIZE
                        )
                    else:
                        clip_window = None
                    green_data = (
                        green_src.read(1, window=clip_window)
                        if clip_window
                        else green_src.read(1)
                    )
                elif aoi_geometry is not None:
                    green_data, transform = mask(green_src, [aoi_geometry], crop=True)
                    green_data = green_data[0]
                else:
                    green_data = green_src.read(1)
                profile = green_src.profile.copy()
                crs = green_src.crs

            with rasterio.open(nir_band_asset.href) as nir_src:
                if aoi_geometry is not None:
                    nir_data, _ = mask(nir_src, [aoi_geometry], crop=True)
                    nir_data = nir_data[0]
                elif use_windowed_read:
                    nir_data = (
                        nir_src.read(1, window=clip_window)
                        if clip_window
                        else nir_src.read(1)
                    )
                else:
                    nir_data = nir_src.read(1)

        green_data = green_data.astype(np.float32)
        nir_data = nir_data.astype(np.float32)

        if green_data.shape != nir_data.shape:
            raise ValueError(
                f"Band shapes do not match: Green {green_data.shape} vs NIR {nir_data.shape}"
            )

        print(
            f"\nSentinel-2 bands loaded: Green {green_data.shape}, NIR {nir_data.shape}"
        )
        print(f"CRS: {crs}")

except Exception as e:
    print(f"Error reading band data: {e}")
    raise

## Step 6: Perform Water Detection

Apply sensor-specific water detection method:
- **Sentinel-1**: Convert to dB scale and threshold backscatter (< -15 dB)
- **Sentinel-2**: Calculate NDWI and threshold (> 0.0)

In [ ]:
try:
    if sensor_type == "S1":
        # Sentinel-1: Water detection using backscatter threshold
        # Convert to dB scale: 10 * log10(backscatter)
        # Avoid log(0) by adding small epsilon
        epsilon = 1e-10
        vv_db = 10 * np.log10(np.maximum(vv_data, epsilon))

        # Water threshold: typically < -15 dB for water
        # Lower threshold = more water detected (more sensitive)
        water_threshold_db = -15.0
        water_mask = vv_db < water_threshold_db

        # Create water probability/confidence (inverse of backscatter)
        # Lower backscatter = higher water probability
        vv_normalized = np.clip(
            (vv_db - np.nanmin(vv_db)) / (np.nanmax(vv_db) - np.nanmin(vv_db)), 0, 1
        )
        water_probability = (
            1 - vv_normalized
        )  # Invert so low backscatter = high probability

        print(f"\nSentinel-1 water detection complete!")
        print(
            f"VV backscatter range: {np.nanmin(vv_db):.2f} to {np.nanmax(vv_db):.2f} dB"
        )
        print(f"Water threshold: < {water_threshold_db} dB")
        print(
            f"Water pixels: {np.sum(water_mask):,} ({np.sum(water_mask) / water_mask.size * 100:.2f}%)"
        )

    else:  # Sentinel-2
        # Sentinel-2: Water detection using NDWI
        # Calculate NDWI: (Green - NIR) / (Green + NIR)
        ndwi_denom = green_data + nir_data
        ndwi_valid = ndwi_denom != 0
        ndwi = np.full_like(green_data, np.nan, dtype=np.float32)
        ndwi[ndwi_valid] = (green_data[ndwi_valid] - nir_data[ndwi_valid]) / ndwi_denom[
            ndwi_valid
        ]
        ndwi = np.clip(ndwi, -1.0, 1.0)

        # Water threshold: typically > 0.0 or > 0.3
        # Higher threshold = less water detected (more conservative)
        water_threshold_ndwi = 0.0
        water_mask = ndwi > water_threshold_ndwi

        # Use NDWI as water probability (normalize to 0-1)
        water_probability = np.clip((ndwi + 1) / 2, 0, 1)  # Map from [-1, 1] to [0, 1]

        print(f"\nSentinel-2 water detection complete!")
        print(f"NDWI range: {np.nanmin(ndwi):.4f} to {np.nanmax(ndwi):.4f}")
        print(f"Water threshold: > {water_threshold_ndwi}")
        print(
            f"Water pixels: {np.sum(water_mask):,} ({np.sum(water_mask) / water_mask.size * 100:.2f}%)"
        )

except Exception as e:
    print(f"Error performing water detection: {e}")
    raise

## Step 7: Visualize Results

Create visualizations showing the original image and water detection results.

In [ ]:
# Create figure with subplots
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

if sensor_type == "S1":
    # Sentinel-1 visualization
    # Original VV backscatter (dB)
    im1 = axes[0].imshow(vv_db, cmap="gray", vmin=-25, vmax=0)
    axes[0].set_title("VV Backscatter (dB)", fontsize=14, fontweight="bold")
    axes[0].axis("off")
    plt.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04, label="dB")

    # Water probability
    im2 = axes[1].imshow(water_probability, cmap="Blues", vmin=0, vmax=1)
    axes[1].set_title("Water Probability", fontsize=14, fontweight="bold")
    axes[1].axis("off")
    plt.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04, label="Probability")

    # Water mask (binary)
    im3 = axes[2].imshow(
        water_mask.astype(np.uint8),
        cmap=ListedColormap(["#2C3E50", "#3498DB"]),
        vmin=0,
        vmax=1,
    )
    axes[2].set_title("Water Mask", fontsize=14, fontweight="bold")
    axes[2].axis("off")

else:  # Sentinel-2
    # Create RGB composite
    rgb_composite = np.stack(
        [
            np.clip(red_data / np.percentile(red_data[~np.isnan(red_data)], 98), 0, 1)
            if "red_data" in locals()
            else np.clip(
                nir_data / np.percentile(nir_data[~np.isnan(nir_data)], 98), 0, 1
            ),
            np.clip(
                green_data / np.percentile(green_data[~np.isnan(green_data)], 98), 0, 1
            ),
            np.clip(
                blue_data / np.percentile(blue_data[~np.isnan(blue_data)], 98), 0, 1
            )
            if "blue_data" in locals()
            else np.clip(
                green_data / np.percentile(green_data[~np.isnan(green_data)], 98), 0, 1
            ),
        ],
        axis=-1,
    )

    # RGB Composite
    axes[0].imshow(rgb_composite)
    axes[0].set_title("RGB Composite", fontsize=14, fontweight="bold")
    axes[0].axis("off")

    # NDWI
    im2 = axes[1].imshow(ndwi, cmap="Blues", vmin=-1, vmax=1)
    axes[1].set_title("NDWI", fontsize=14, fontweight="bold")
    axes[1].axis("off")
    plt.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04, label="NDWI")

    # Water mask (binary)
    im3 = axes[2].imshow(
        water_mask.astype(np.uint8),
        cmap=ListedColormap(["#2C3E50", "#3498DB"]),
        vmin=0,
        vmax=1,
    )
    axes[2].set_title("Water Mask", fontsize=14, fontweight="bold")
    axes[2].axis("off")

# Add legend for water mask
from matplotlib.patches import Patch

legend_elements = [
    Patch(facecolor="#2C3E50", label="Non-water"),
    Patch(facecolor="#3498DB", label="Water"),
]
axes[2].legend(handles=legend_elements, loc="upper right")

plt.suptitle(
    f"Water Detection - {item.id} ({sensor_type})",
    fontsize=16,
    fontweight="bold",
    y=1.02,
)
plt.tight_layout()
plt.show()

# Print statistics
print("\nWater Detection Statistics:")
total_pixels = water_mask.size
water_pixels = np.sum(water_mask)
water_percentage = (water_pixels / total_pixels) * 100
print(f"  Total pixels: {total_pixels:,}")
print(f"  Water pixels: {water_pixels:,} ({water_percentage:.2f}%)")
print(
    f"  Non-water pixels: {total_pixels - water_pixels:,} ({100 - water_percentage:.2f}%)"
)

print("\nVisualization complete!")

## Summary

This notebook has successfully:

1. ✅ Loaded the STAC item from: `{{STAC_ITEM_LINK}}`
2. ✅ Detected sensor type: **{{SENSOR_TYPE}}**
3. ✅ Determined area of interest (AOI or default window)
4. ✅ Accessed appropriate bands for water detection
5. ✅ Applied sensor-specific water detection method
6. ✅ Generated water mask and probability map
7. ✅ Visualized the results

### Detection Method Used

**{{SENSOR_TYPE}}**: {{METHOD_DESCRIPTION}}

### Next Steps

You can now:
- Export the water mask as a GeoTIFF
- Adjust thresholds for different sensitivity levels
- Compare water detection across different dates
- Combine Sentinel-1 and Sentinel-2 results for improved accuracy
- Calculate water area statistics

### Resources

- **Collection**: {{STAC_COLLECTION_NAME}}
- **Sensor**: {{SENSOR_TYPE}}

---

*This notebook was automatically generated from a template and configured with your selected dataset.*